###1. Create a SparkSession: Write the basic code to initialize a SparkSession with a specific app name. 

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("DataEngineeringJob") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.executor.memory", "2g") \
    .getOrCreate()

In [0]:
spark.stop()

###2. Read CSV with Header: Read a CSV file while inferring the schema and specifying that the first row is a header.

In [0]:
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/Volumes/dbx/scenarios/files/annual-enterprise-survey.csv")

df.show()
df.printSchema()

###3. Select & Rename: Select three columns from a DataFrame and rename one of them using alias().

In [0]:
# Sample DataFrame
data = [
    (1, "Bharath", 28, "India"),
    (2, "Ravi", 30, "USA")
]

columns = ["id", "name", "age", "country"]

df = spark.createDataFrame(data, columns)

# Select three columns and rename one using alias()
df_selected = df.select(
    "id",
    df.name.alias("full_name"),   # Renamed column
    "age"
)

df_selected.show()

###4. Filter with Multiple Conditions: Filter rows where salary > 5000 and department == 'Sales'

In [0]:
from pyspark.sql.functions import col

# Sample DataFrame
data = [
    (1, "Bharath", 6000, "Sales"),
    (2, "Ravi", 4500, "Sales"),
    (3, "Anu", 7000, "HR"),
    (4, "John", 8000, "Sales")
]

columns = ["id", "name", "salary", "department"]

df = spark.createDataFrame(data, columns)

# Filter where salary > 5000 AND department == 'Sales'
filtered_df = df.filter(
    (col("salary") > 5000) & (col("department") == "Sales")
)

filtered_df.show()

###5. Add a New Column: Create a new column total_price by multiplying quantity and unit_price

In [0]:
from pyspark.sql.functions import col

# Sample DataFrame
data = [
    (1, 2, 500),
    (2, 5, 200),
    (3, 3, 1000)
]

columns = ["product_id", "quantity", "unit_price"]

df = spark.createDataFrame(data, columns)

# Add new column total_price
df_new = df.withColumn(
    "total_price",
    col("quantity") * col("unit_price")
)

df_new.show()

###6. Remove Duplicates: Use dropDuplicates() to remove rows based on specific primary key columns. 

In [0]:
# Sample DataFrame
data = [
    (1, "Bharath", "Sales"),
    (2, "Ravi", "HR"),
    (1, "Bharath", "Sales"),   # Duplicate row
    (3, "Anu", "IT"),
    (2, "Ravi", "HR")          # Duplicate row
]

columns = ["emp_id", "name", "department"]

df = spark.createDataFrame(data, columns)

# Remove duplicates based on primary key column (emp_id)
df_dedup = df.dropDuplicates(["emp_id"])

df_dedup.show()

###7. Sort Data: Sort a DataFrame by date in descending order. 

In [0]:
from pyspark.sql.functions import col

# Sample DataFrame
data = [
    (1, "2024-01-10"),
    (2, "2024-03-15"),
    (3, "2023-12-01")
]

columns = ["id", "order_date"]

df = spark.createDataFrame(data, columns)

# Convert string to date (recommended if column is string)
from pyspark.sql.functions import to_date
df = df.withColumn("order_date", to_date("order_date", "yyyy-MM-dd"))

# Sort by date in descending order
df_sorted = df.orderBy(col("order_date").desc())

df_sorted.show()

###8. Cast Data Types: Convert a string column representing a date into a TimestampType. 

In [0]:
from pyspark.sql.functions import to_timestamp, col

# Sample DataFrame
data = [
    (1, "2024-01-10 14:30:00"),
    (2, "2024-03-15 09:45:10")
]

columns = ["id", "event_time"]

df = spark.createDataFrame(data, columns)

# Convert string column to TimestampType
df_casted = df.withColumn(
    "event_time",
    to_timestamp(col("event_time"), "yyyy-MM-dd HH:mm:ss")
)

df_casted.printSchema()
df_casted.show()

###9. Union DataFrames: Combine two DataFrames with the same schema using unionByName(). 

In [0]:
# First DataFrame
data1 = [
    (1, "Bharath", 28),
    (2, "Ravi", 30)
]

columns = ["id", "name", "age"]

df1 = spark.createDataFrame(data1, columns)

# Second DataFrame (same schema)
data2 = [
    (3, "Anu", 25),
    (4, "John", 32)
]

df2 = spark.createDataFrame(data2, columns)

# Union using column names
df_union = df1.unionByName(df2)

df_union.show()

###	10. Handle Nulls: Fill all null values in a column with a default value like 0 or Unknown. 


In [0]:
# Sample DataFrame
data = [
    (1, 5000, "Sales"),
    (2, None, "HR"),
    (3, 7000, None)
]

columns = ["emp_id", "salary", "department"]

df = spark.createDataFrame(data, columns)

# Fill null salary with 0
df_filled_salary = df.fillna({"salary": 0})

# Fill null department with "Unknown"
df_filled_all = df_filled_salary.fillna({"department": "Unknown"})

df_filled_all.show()

###11. Group By & Aggregate: Calculate the average salary per department. 

In [0]:
from pyspark.sql.functions import avg

# Sample DataFrame
data = [
    (1, "Sales", 5000),
    (2, "Sales", 7000),
    (3, "HR", 4000),
    (4, "HR", 6000),
    (5, "IT", 8000)
]

columns = ["emp_id", "department", "salary"]

df = spark.createDataFrame(data, columns)

# Group by department and calculate average salary
df_avg = df.groupBy("department") \
           .agg(avg("salary").alias("avg_salary"))

df_avg.show()

###12. Multiple Aggregations: Find the min, max, and count of a column in one groupBy operation. 

In [0]:
from pyspark.sql.functions import min, max, count

# Sample DataFrame
data = [
    (1, "Sales", 5000),
    (2, "Sales", 7000),
    (3, "HR", 4000),
    (4, "HR", 6000),
    (5, "IT", 8000)
]

columns = ["emp_id", "department", "salary"]

df = spark.createDataFrame(data, columns)

# Group by department and perform multiple aggregations
df_agg = df.groupBy("department").agg(
    min("salary").alias("min_salary"),
    max("salary").alias("max_salary"),
    count("salary").alias("salary_count")
)

df_agg.show()

###13. Inner Join: Join two DataFrames on an employee_id column.

In [0]:
# Employees DataFrame
emp_data = [
    (1, "Bharath", "Sales"),
    (2, "Ravi", "HR"),
    (3, "Anu", "IT")
]

emp_columns = ["employee_id", "name", "department"]
df_emp = spark.createDataFrame(emp_data, emp_columns)

# Salary DataFrame
salary_data = [
    (1, 6000),
    (2, 5000),
    (4, 7000)   # No matching employee
]

salary_columns = ["employee_id", "salary"]
df_salary = spark.createDataFrame(salary_data, salary_columns)

# Inner Join on employee_id
df_joined = df_emp.join(
    df_salary,
    on="employee_id",
    how="inner"
)

df_joined.show()

###14. Left Join: Perform a left join and filter for records that didn't have a match in the right table. 

In [0]:
from pyspark.sql.functions import col

# Employees DataFrame (Left table)
emp_data = [
    (1, "Bharath"),
    (2, "Ravi"),
    (3, "Anu"),
    (4, "John")
]

emp_columns = ["employee_id", "name"]
df_emp = spark.createDataFrame(emp_data, emp_columns)

# Salary DataFrame (Right table)
salary_data = [
    (1, 6000),
    (2, 5000)
]

salary_columns = ["employee_id", "salary"]
df_salary = spark.createDataFrame(salary_data, salary_columns)

# Left Join
df_left = df_emp.join(
    df_salary,
    on="employee_id",
    how="left"
)

# Filter records that did NOT have a match in right table
df_no_match = df_left.filter(col("salary").isNull())

df_no_match.show()

###15. Pivot Table: Write code to pivot a "sales" column by "year". 

In [0]:
from pyspark.sql.functions import sum

# Sample DataFrame
data = [
    ("ProductA", 2022, 1000),
    ("ProductA", 2023, 1500),
    ("ProductB", 2022, 2000),
    ("ProductB", 2023, 2500)
]

columns = ["product", "year", "sales"]

df = spark.createDataFrame(data, columns)

# Pivot sales by year
df_pivot = df.groupBy("product") \
             .pivot("year") \
             .agg(sum("sales"))

df_pivot.show()

###16. Unpivot (Stack): Convert multiple columns (e.g., Q1, Q2, Q3 sales) into rows. 

In [0]:
from pyspark.sql.functions import expr

# Sample DataFrame
data = [
    ("ProductA", 1000, 1500, 2000),
    ("ProductB", 2000, 2500, 3000)
]

columns = ["product", "Q1", "Q2", "Q3"]

df = spark.createDataFrame(data, columns)

# Unpivot using stack()
df_unpivot = df.select(
    "product",
    expr("""
        stack(3, 
              'Q1', Q1,
              'Q2', Q2,
              'Q3', Q3
        ) as (quarter, sales)
    """)
)

df_unpivot.show()

###17. Self Join: Use a self-join to find pairs of employees working in the same department. 

In [0]:
from pyspark.sql.functions import col

# Sample DataFrame
data = [
    (1, "Bharath", "Sales"),
    (2, "Ravi", "Sales"),
    (3, "Anu", "HR"),
    (4, "John", "HR"),
    (5, "Mike", "IT")
]

columns = ["employee_id", "name", "department"]

df = spark.createDataFrame(data, columns)

# Create aliases for self join
df1 = df.alias("e1")
df2 = df.alias("e2")

# Self join on department and avoid duplicate/self pairing
df_pairs = df1.join(
    df2,
    (col("e1.department") == col("e2.department")) &
    (col("e1.employee_id") < col("e2.employee_id")),
    "inner"
).select(
    col("e1.employee_id").alias("emp1_id"),
    col("e1.name").alias("emp1_name"),
    col("e2.employee_id").alias("emp2_id"),
    col("e2.name").alias("emp2_name"),
    col("e1.department")
)

df_pairs.show()

###18. Join Optimization (Broadcast): Manually trigger a broadcast join for a small lookup table. 

- Use broadcast when one table is small (lookup/dimension table).
- Small table is sent to all executors.
- Avoids shuffle.
- Improves join performance for large-to-small joins.
- Spark automatically broadcasts small tables (based on spark.sql.autoBroadcastJoinThreshold), but broadcast() forces it manually.

In [0]:
from pyspark.sql.functions import broadcast

# Large DataFrame (Fact table)
fact_data = [
    (1, 101, 5000),
    (2, 102, 7000),
    (3, 103, 6000)
]

fact_columns = ["transaction_id", "dept_id", "amount"]
df_fact = spark.createDataFrame(fact_data, fact_columns)

# Small Lookup DataFrame (Dimension table)
lookup_data = [
    (101, "Sales"),
    (102, "HR"),
    (103, "IT")
]

lookup_columns = ["dept_id", "department"]
df_lookup = spark.createDataFrame(lookup_data, lookup_columns)

# Broadcast Join
df_joined = df_fact.join(
    broadcast(df_lookup),   # Force broadcast
    on="dept_id",
    how="inner"
)

df_joined.show()

###19. Anti Join: Find records in Table A that do not exist in Table B. 

In [0]:
# Table A
data_a = [
    (1, "Bharath"),
    (2, "Ravi"),
    (3, "Anu"),
    (4, "John")
]

columns_a = ["employee_id", "name"]
df_a = spark.createDataFrame(data_a, columns_a)

# Table B
data_b = [
    (1,),
    (3,)
]

columns_b = ["employee_id"]
df_b = spark.createDataFrame(data_b, columns_b)

# Left Anti Join
df_anti = df_a.join(
    df_b,
    on="employee_id",
    how="left_anti"
)

df_anti.show()

###20. Cross Join Warning: Write a cross-join but explain why it’s generally avoided. 

- Produces Cartesian product (rows_A × rows_B).
- Causes massive data explosion for large tables.
- Leads to heavy shuffle, memory pressure, and OOM errors.
- Can severely degrade cluster performance.
- Use only when Dataset is very small
- You intentionally need all possible combinations
- No join condition exists by design.

In [0]:
# DataFrame 1
data1 = [
    (1, "A"),
    (2, "B")
]

df1 = spark.createDataFrame(data1, ["id", "category"])

# DataFrame 2
data2 = [
    ("X",),
    ("Y",)
]

df2 = spark.createDataFrame(data2, ["type"])

# Enable cross join if disabled
spark.conf.set("spark.sql.crossJoin.enabled", "true")

# Cross Join
df_cross = df1.crossJoin(df2)

df_cross.show()